# Hardware 2-qubit scheduling-policy comparison

This notebook runs a Bell-pair Heisenberg workload on real QUBEX hardware and compares the modern Qiskit-timed execution path with the deprecated `legacy_device_gateway` timing policy. The hardware experiment is initialized with `bell_state.DEFAULT_QUBIT_LABELS`, while the logical 2-qubit workload is placed on `bell_state.DEFAULT_BELL_PAIR`.

The same circuit family is used for three curves:

1. ideal local simulation,
2. QUBEX hardware with `timing_policy="qiskit"` and ALAP scheduling,
3. QUBEX hardware with `timing_policy="legacy_device_gateway"`.

The observable is staggered magnetization `(Z0 - Z1) / 2` after starting from `|01>`. `READOUT_MITIGATION=True` enables QUBEX classifier confusion-matrix mitigation through the provider's `backend.run(..., readout_mitigation=True)` option. The default uses `LAYERS = 1`; on the current 144Qv2 pair, `LAYERS = 3` decomposes to 18 CX gates and the algorithmic error dominates the comparison.

By default this notebook connects to hardware. Set `RUN_ON_HARDWARE = False` if you only want compilation, schedule validation, and the ideal simulation.


In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import SVG, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

# Re-running this cell in an existing notebook kernel should pick up local provider edits.
for module_name in (
    "qiskit_qubex_provider.dynamical_decoupling",
    "qiskit_qubex_provider.executor",
    "qiskit_qubex_provider.backend",
    "qiskit_qubex_provider.provider",
    "qiskit_qubex_provider",
):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

from qiskit_qubex_provider import (
    QubexProvider,
    build_pulse_schedule_timeline_figure,
    diff_pulse_schedules,
    summarize_pulse_schedule,
)

HERE = Path.cwd()
if not (HERE / "bell_state.py").exists():
    HERE = Path("examples/hardware").resolve()

spec = importlib.util.spec_from_file_location("hardware_bell_state", HERE / "bell_state.py")
bell_state = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(bell_state)


In [ ]:
DEVICE_ID = bell_state.DEFAULT_DEVICE_ID
QUBIT_LABELS = bell_state.DEFAULT_QUBIT_LABELS
BELL_PAIR = bell_state.DEFAULT_BELL_PAIR
WORKLOAD_QUBIT_LABELS = BELL_PAIR
CONFIG_ROOT = HERE / "qubex-config"
OUTPUT_DIR = HERE / "generated"

NUM_QUBITS = len(WORKLOAD_QUBIT_LABELS)
SHOTS = 4096
RUN_ON_HARDWARE = True
READOUT_MITIGATION = True

N_STEPS = 1
TOTAL_TIMES = np.linspace(0, 5, 30)
TOTAL_TIMES_IDEAL = np.linspace(0, 5, 241)
SAMPLE_TOTAL_TIME = 2.5
REFERENCE_J_VALUES = np.array([0.542641286533492])
REFERENCE_EXACT_PROBABILITY = np.array([
    1.0000000000000004,
    0.9740452798096243,
    0.8997739059060192,
    0.7874669061709083,
    0.6526703966232,
    0.514043607598628,
    0.3907759765373158,
    0.2999308467604735,
    0.2540834711961876,
    0.2595802808003742,
    0.31566037874521213,
    0.4145608676951965,
    0.5425914302242566,
    0.682029413392349,
    0.8135730900686025,
    0.9190135038746419,
    0.9837550471459701,
    0.9988358604676509,
    0.962168379626311,
    0.8788283070503398,
    0.760352006816297,
    0.6231395814402649,
    0.4861846848277263,
    0.36844532242017575,
    0.2862195859865792,
    0.25088958742870654,
    0.26734588795666897,
    0.3333105213007486,
    0.43965232158944606,
    0.5716509065772457,
])
REFERENCE_TROTTER_PROBABILITY = np.array([
    0.9999999999999996,
    0.9740452798096245,
    0.899773905906018,
    0.7874669061709073,
    0.6526703966232,
    0.5140436075986273,
    0.3907759765373152,
    0.2999308467604731,
    0.2540834711961876,
    0.2595802808003743,
    0.31566037874521213,
    0.41456086769519623,
    0.5425914302242565,
    0.6820294133923483,
    0.8135730900686025,
    0.9190135038746412,
    0.9837550471459696,
    0.99883586046765,
    0.9621683796263096,
    0.87882830705034,
    0.7603520068162962,
    0.6231395814402645,
    0.4861846848277257,
    0.36844532242017564,
    0.2862195859865792,
    0.2508895874287062,
    0.26734588795666886,
    0.3333105213007488,
    0.4396523215894458,
    0.5716509065772457,
])

TOPOLOGY_PATH = bell_state.topology_json_path(
    output_dir=OUTPUT_DIR,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
)
compile_experiment = bell_state.make_experiment(
    config_root=CONFIG_ROOT,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
)
bell_state.generate_device_topology(
    config_root=CONFIG_ROOT,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
    bell_pair=BELL_PAIR,
    output_path=TOPOLOGY_PATH,
    pulse_source=compile_experiment,
)

def provider_from_experiment(experiment, *, timing_policy: str) -> QubexProvider:
    return QubexProvider.from_experiment(
        experiment,
        name=f"{DEVICE_ID}-{timing_policy}",
        device_topology=TOPOLOGY_PATH,
        qubit_labels=QUBIT_LABELS,
        timing_policy=timing_policy,
        native=True,
        execute_options={
            "state_classification": False,
            "time_integration": True,
            "plot": False,
        },
    )

provider_qiskit = provider_from_experiment(compile_experiment, timing_policy="qiskit")
provider_legacy = provider_from_experiment(compile_experiment, timing_policy="legacy_device_gateway")
backend_qiskit = provider_qiskit.get_backend()
backend_legacy = provider_legacy.get_backend()
backend_ideal = QubexProvider(num_qubits=NUM_QUBITS, coupling_map=[(0, 1)]).get_backend()
initial_layout = bell_state.bell_initial_layout(
    qubit_labels=QUBIT_LABELS,
    bell_pair=BELL_PAIR,
)

print("qiskit backend:", backend_qiskit.name)
print("legacy backend:", backend_legacy.name)
print("topology:", TOPOLOGY_PATH)
print("topology svg:", TOPOLOGY_PATH.with_suffix(".svg"))
print("experiment qubits:", QUBIT_LABELS)
print("logical Heisenberg qubits:", WORKLOAD_QUBIT_LABELS)
print("J values:", REFERENCE_J_VALUES)
print("shots:", SHOTS)
print("run on hardware:", RUN_ON_HARDWARE)
print("readout mitigation:", READOUT_MITIGATION)


In [ ]:
display(SVG(filename=str(TOPOLOGY_PATH.with_suffix(".svg"))))


## Bell-pair Heisenberg workload

The model starts from `|01>` on `WORKLOAD_QUBIT_LABELS` and applies `LAYERS` first-order Heisenberg steps. Keep `LAYERS = 1` for the hardware comparison first; increasing it is useful only after the short-depth curve follows the ideal reference. The total angle `theta` is split across layers:

`exp(-i step XX / 2) exp(-i step YY / 2) exp(-i step ZZ / 2)`

The measured observable is staggered magnetization. Values near `-1` mean the state still looks like `|01>`, while values near `+1` mean it has moved toward `|10>`.


In [ ]:
def create_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def uncompute_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def add_xz_pauli_rotation(qc: QuantumCircuit, q0: int, q1: int, angle: float) -> None:
    qc.h(q0)
    qc.rzz(angle, q0, q1)
    qc.h(q0)


def add_heisenberg_interactions(
    qc: QuantumCircuit,
    edge: tuple[int, int],
    coupling: float,
    evolution_time: float,
) -> None:
    control, target = edge
    theta = float(coupling) * evolution_time
    qc.cx(control, target)
    qc.rx(2.0 * theta, control)
    qc.rz(2.0 * theta, target)
    add_xz_pauli_rotation(qc, control, target, -2.0 * theta)
    qc.cx(control, target)


HEISENBERG_EDGE = (0, 1)


def heisenberg_2q_circuit(total_time: float, *, n_steps: int = N_STEPS) -> QuantumCircuit:
    qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
    create_half_ghz_state(qc)

    step_time = float(total_time) / n_steps
    for _ in range(n_steps):
        add_heisenberg_interactions(qc, HEISENBERG_EDGE, REFERENCE_J_VALUES[0], step_time)

    uncompute_half_ghz_state(qc)
    qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))
    return qc


def probability_zero_from_counts(counts: dict[str, int]) -> float:
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return counts.get("0" * NUM_QUBITS, 0) / total


def exact_probability_zero(circuit: QuantumCircuit) -> float:
    state_circuit = circuit.remove_final_measurements(inplace=False)
    probabilities = Statevector.from_instruction(state_circuit).probabilities_dict()
    return float(probabilities.get("0" * NUM_QUBITS, 0.0))


def compile_physical(total_time: float, backend) -> QuantumCircuit:
    return transpile(
        heisenberg_2q_circuit(total_time),
        backend,
        initial_layout=initial_layout,
        optimization_level=1,
        seed_transpiler=7,
    )


def compile_qiskit(
    total_time: float,
    *,
    scheduling_method: str = "alap",
) -> QuantumCircuit:
    physical = compile_physical(total_time, backend_qiskit)
    return transpile(
        physical,
        backend_qiskit,
        scheduling_method=scheduling_method,
        optimization_level=0,
    )


def compile_legacy(total_time: float) -> QuantumCircuit:
    return compile_physical(total_time, backend_legacy)

heisenberg_2q_circuit(SAMPLE_TOTAL_TIME).draw("text")


## Ideal simulation reference

The local simulator does not model device noise. It gives a reference curve for the hardware measurements.


In [ ]:
ideal = [
    exact_probability_zero(heisenberg_2q_circuit(float(total_time)))
    for total_time in TOTAL_TIMES_IDEAL
]
reference_trotter = REFERENCE_TROTTER_PROBABILITY
reference_exact = REFERENCE_EXACT_PROBABILITY

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal,
    name="trotter circuit statevector",
    line=dict(color="#475569", dash="dot"),
))
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES,
    y=reference_exact,
    name="reference exact simulation",
    mode="lines+markers",
    line=dict(color="#2563eb"),
))
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES,
    y=reference_trotter,
    name="reference trotter simulation",
    mode="markers",
    marker=dict(color="#dc2626", symbol="x"),
))
fig.update_layout(
    title="2-qubit Heisenberg ideal references",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="probability of |00>", range=[-0.05, 1.05]),
    width=760,
    height=430,
)
fig.show()

print("statevector min:", float(np.min(ideal)))
print("reference exact min:", float(np.min(reference_exact)))
print("reference trotter min:", float(np.min(reference_trotter)))


## Schedule comparison

`qiskit` consumes the Qiskit scheduled circuit, including explicit start times and explicit readout pulses. `legacy_device_gateway` rebuilds the pulse program through the old sequential path and uses `final_measurement=True` at execution time, so its validated pulse schedule excludes the final readout window. The timelines are useful for seeing the construction difference, but the measured observable is the main comparison.


In [ ]:
QISKIT_COMPILE_VARIANTS = {
    "qiskit ALAP": dict(scheduling_method="alap"),
    "qiskit ASAP": dict(scheduling_method="asap"),
}

sample_circuits = {
    name: compile_qiskit(SAMPLE_TOTAL_TIME, **options)
    for name, options in QISKIT_COMPILE_VARIANTS.items()
}
sample_circuits["legacy_device_gateway"] = compile_legacy(SAMPLE_TOTAL_TIME)

sample_schedules = {
    name: backend_qiskit.validate(circuit)[0]
    for name, circuit in sample_circuits.items()
    if name.startswith("qiskit")
}
sample_schedules["legacy_device_gateway"] = backend_legacy.validate(sample_circuits["legacy_device_gateway"])[0]

for name, schedule in sample_schedules.items():
    print(f"{name}")
    print(summarize_pulse_schedule(schedule))

print()
print("qiskit ALAP vs ASAP diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["qiskit ASAP"]))

print()
print("qiskit ALAP vs legacy diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["legacy_device_gateway"]))


In [ ]:
for name in ("qiskit ALAP", "qiskit ASAP"):
    build_pulse_schedule_timeline_figure(
        sample_schedules[name],
        title=f"2q Heisenberg sample: {name}",
    ).show()


In [ ]:
build_pulse_schedule_timeline_figure(
    sample_schedules["legacy_device_gateway"],
    title="2q Heisenberg sample: legacy_device_gateway",
).show()


In [ ]:
qiskit_circuits_by_variant = {
    name: [compile_qiskit(float(total_time), **options) for total_time in TOTAL_TIMES]
    for name, options in QISKIT_COMPILE_VARIANTS.items()
}
legacy_circuits = [compile_legacy(float(total_time)) for total_time in TOTAL_TIMES]

# Keep the original variable name for quick ad-hoc inspection.
qiskit_circuits = qiskit_circuits_by_variant["qiskit ALAP"]


## Hardware sweep: simulation vs qiskit vs legacy

This cell connects to hardware, builds classifiers once, then runs both circuit batches. `READOUT_MITIGATION=True` applies QUBEX's inverse confusion matrix during provider result conversion. The legacy executor does not emit explicit readout pulses for Qiskit `measure`, so the legacy hardware run sets `final_measurement=True`.


In [ ]:
hardware_sweep = None
if RUN_ON_HARDWARE:
    experiment = bell_state.make_experiment(
        config_root=CONFIG_ROOT,
        device_id=DEVICE_ID,
        qubit_labels=QUBIT_LABELS,
    )
    try:
        experiment.connect()
        execute_provider_qiskit = provider_from_experiment(experiment, timing_policy="qiskit")
        execute_provider_legacy = provider_from_experiment(experiment, timing_policy="legacy_device_gateway")
        execute_provider_qiskit.build_classifier(targets=list(BELL_PAIR), shots=SHOTS)
        execute_backend_qiskit = execute_provider_qiskit.get_backend()
        execute_backend_legacy = execute_provider_legacy.get_backend()

        hardware_sweep = {}
        for name, circuits in qiskit_circuits_by_variant.items():
            result = execute_backend_qiskit.run(
                circuits,
                shots=SHOTS,
                readout_mitigation=READOUT_MITIGATION,
            ).result()
            hardware_sweep[name] = [
                probability_zero_from_counts(result.get_counts(i))
                for i in range(len(circuits))
            ]

        legacy_result = execute_backend_legacy.run(
            legacy_circuits,
            shots=SHOTS,
            final_measurement=True,
            readout_mitigation=READOUT_MITIGATION,
        ).result()
        hardware_sweep["legacy_device_gateway"] = [
            probability_zero_from_counts(legacy_result.get_counts(i))
            for i in range(len(legacy_circuits))
        ]
        print(hardware_sweep)
    finally:
        experiment.disconnect()
else:
    print("RUN_ON_HARDWARE is False; hardware sweep was skipped.")


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal,
    name="trotter circuit statevector",
    line=dict(color="#475569", dash="dot"),
))
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES,
    y=reference_exact,
    name="reference exact simulation",
    mode="lines+markers",
    line=dict(color="#2563eb"),
))
if hardware_sweep is not None:
    for name, values in hardware_sweep.items():
        fig.add_trace(go.Scatter(
            x=TOTAL_TIMES,
            y=values,
            mode="lines+markers",
            name=f"{name} hardware",
        ))
fig.update_layout(
    title="2-qubit Heisenberg: simulation vs hardware timing policies",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="probability of |00>", range=[-0.05, 1.05]),
    width=760,
    height=430,
)
fig.show()
